In [5]:
import warnings
import itertools
import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.express as px

In [8]:
data =sm.datasets.co2.load_pandas()
y = data.data
y = y['co2'].resample('MS').mean()
y = y.fillna(y.bfill())
px.line(y)

In [9]:
p=d=q=range(0,2)
#生成季节参数组合，专门给SARIMA使用
pdq = list(itertools.product(p,d,q))
seasonal_pdq = [(x[0],x[1],x[2],12) for x in pdq]
print('SARIMA参数组合：')
print(f'SARIMA:{pdq[1]} x {seasonal_pdq[1]}')
print(f'SARIMA:{pdq[1]} x {seasonal_pdq[2]}')
print(f'SARIMA:{pdq[2]} x {seasonal_pdq[3]}')
print(f'SARIMA:{pdq[2]} x {seasonal_pdq[4]}')

warnings.filterwarnings('ignore')

for param in pdq:
    for param_seasonal in seasonal_pdq:
        try:
            mod = sm.tsa.statespace.SARIMAX(y,
                                            order=param,
                                            seasonal_order=param_seasonal,
                                            enforce_stationarity=False,
                                            enforce_invertibility=False)
            results = mod.fit()
            print('ARIMA{}x{}12 - AIC:{}'.format(param,param_seasonal,results.aic))
        except:
            continue

SARIMA参数组合：
SARIMA:(0, 0, 1) x (0, 0, 1, 12)
SARIMA:(0, 0, 1) x (0, 1, 0, 12)
SARIMA:(0, 1, 0) x (0, 1, 1, 12)
SARIMA:(0, 1, 0) x (1, 0, 0, 12)
ARIMA(0, 0, 0)x(0, 0, 0, 12)12 - AIC:7612.583429881011
ARIMA(0, 0, 0)x(0, 0, 1, 12)12 - AIC:6787.3436240358005
ARIMA(0, 0, 0)x(0, 1, 0, 12)12 - AIC:1854.8282341411787
ARIMA(0, 0, 0)x(0, 1, 1, 12)12 - AIC:1596.7111727634306
ARIMA(0, 0, 0)x(1, 0, 0, 12)12 - AIC:1058.9388921320046
ARIMA(0, 0, 0)x(1, 0, 1, 12)12 - AIC:1056.2878457196653
ARIMA(0, 0, 0)x(1, 1, 0, 12)12 - AIC:1361.657897807711
ARIMA(0, 0, 0)x(1, 1, 1, 12)12 - AIC:1044.7647912875404
ARIMA(0, 0, 1)x(0, 0, 0, 12)12 - AIC:6881.048755640596
ARIMA(0, 0, 1)x(0, 0, 1, 12)12 - AIC:6072.6623276349565
ARIMA(0, 0, 1)x(0, 1, 0, 12)12 - AIC:1379.1941067434811
ARIMA(0, 0, 1)x(0, 1, 1, 12)12 - AIC:1241.4174716829602
ARIMA(0, 0, 1)x(1, 0, 0, 12)12 - AIC:1088.062814292795
ARIMA(0, 0, 1)x(1, 0, 1, 12)12 - AIC:780.4315988939434
ARIMA(0, 0, 1)x(1, 1, 0, 12)12 - AIC:1119.5957893593477
ARIMA(0, 0, 1)x(1, 1,

In [10]:
mod = sm.tsa.statespace.SARIMAX(y,
                               order=(1,1,1),
                               seasonal_order=(1,1,1,12),
                               enforce_stationarity=False,
                               enforce_invertibility=False)
results = mod.fit()
print(results.summary().tables[1])

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.3182      0.092      3.443      0.001       0.137       0.499
ma.L1         -0.6255      0.077     -8.167      0.000      -0.776      -0.475
ar.S.L12       0.0010      0.001      1.732      0.083      -0.000       0.002
ma.S.L12      -0.8769      0.026    -33.812      0.000      -0.928      -0.826
sigma2         0.0972      0.004     22.633      0.000       0.089       0.106


In [12]:
import plotly.graph_objs as go
#todo:result就是我之前训练的模型
pred = results.get_prediction(start=pd.to_datetime('1998-01-01'),dynamic=False)
pred_ci = pred.conf_int()
trace_observed = go.Scatter(x=y.index,
                            y=y,
                            mode='lines',
                            name='observed')
trace_forecast = go.Scatter(x=pred.predicted_mean.index,
                            y=pred.predicted_mean,
                            mode='lines',
                            name='forecast')
trace_ci = go.Scatter(x=pred_ci.index,
                      y=pred_ci.iloc[:,0],
                      mode='lines',
                      line=dict(color='rgba(255,255,255,0)'),
                      showlegend=False)
trace_ci2 = go.Scatter(x=pred_ci.index,
                      y=pred_ci.iloc[:,1],
                      mode='lines',
                      line=dict(color='rgba(255,255,255,0)'),
                      fillcolor='rgba(0,100,80,0.2)',
                      fill='tonexty',
                      showlegend=False)
layout = go.Layout(title="co2 Levels Forecast with SARIMA",
                   xaxis = dict(title='Date',range=['1990-01-01',pred.predicted_mean.index[-1].strftime('%Y-%m-%d')]),
                   yaxis = dict(title='CO2 Levels',range=[340,380]),
                   legend=dict(x=0.1,y=1.1)
                   )
fig = go.Figure(data=[trace_observed,trace_forecast,trace_ci,trace_ci2],layout=layout)
fig.show()